In [139]:
import pandas as pd
import numpy as np 

In [140]:
dataset1 = pd.read_csv('../data/used_cars_data.csv')
dataset2 = pd.read_csv('../data/used_car_dataset.csv')

Droping Unnessery column from the dataset 1

In [141]:
Drop_variables_dataset1 =['S.No.','Location','Engine','Power','Seats','New_Price','Mileage']
dataset1 = dataset1.drop(Drop_variables_dataset1, axis=1)

Spliting Name as model and brand


In [142]:
dataset1[['Brand','Model']] = dataset1['Name'].str.split(' ',n=1 ,expand = True)
dataset1 =dataset1.drop(['Name'],axis=1)
# rearranging the column 
cols = list(dataset1.columns)
cols.remove('Brand')
cols.remove('Model')
new_order = ['Brand', 'Model'] + cols
dataset1 = dataset1[new_order]

In [143]:
dataset1 = dataset1.iloc[:-1234, :]
#print(dataset1.isna().sum())
dataset1['Brand'] = dataset1['Brand'].replace('Land', 'Land Rover')
dataset1['Brand'] = dataset1['Brand'].replace('Maruti', 'maruti suzuki')

In [144]:
alphabetic_columns = ['Brand','Fuel_Type','Transmission','Owner_Type']
dataset1[alphabetic_columns] = dataset1[alphabetic_columns].map(lambda x: x.lower() if isinstance(x, str) else x)

In [145]:
dataset2 = dataset2.drop(['PostedDate','AdditionInfo','Age'],axis=1)

In [146]:
dataset2['AskPrice'] = dataset2['AskPrice'].str.replace('₹', '', regex=False)
dataset2['AskPrice'] = dataset2['AskPrice'].str.replace(',', '', regex=False)
dataset2['AskPrice'] = dataset2['AskPrice'].str.strip()
dataset2['AskPrice'] = pd.to_numeric(dataset2['AskPrice'], errors='coerce')
dataset1['Price'] = pd.to_numeric(dataset1['Price'],errors='coerce')
dataset2['AskPrice'] = dataset2['AskPrice'] / 100000  # convert to Lakhs
dataset2.rename(columns={'AskPrice': 'Price'}, inplace=True)

In [147]:
dataset2.rename(columns={
    'model': 'Model',
    'kmDriven': 'Kilometers_Driven',
    'FuelType': 'Fuel_Type',
    'Owner': 'Owner_Type'
}, inplace=True)
dataset2[alphabetic_columns] = dataset2[alphabetic_columns].map(lambda x: x.lower() if isinstance(x, str) else x)


In [148]:
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.replace(',', '', regex=False)
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.replace(' km', '', regex=False)
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.strip()
dataset2['Kilometers_Driven'] = pd.to_numeric(dataset2['Kilometers_Driven'], errors='coerce')

In [149]:
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].replace(0,pd.NA)

In [150]:
cols = ['Brand', 'Model', 'Year', 'Kilometers_Driven', 'Transmission', 'Owner_Type', 'Fuel_Type', 'Price']
dataset1_clean =dataset1[cols]
dataset2_clean =dataset2[cols]

dataset_merged = pd.concat([dataset1_clean , dataset2_clean], ignore_index=True)

In [151]:
# 1. Calculate your historical baseline metric once
median_km = dataset_merged['Kilometers_Driven'].median()
dataset_merged['Kilometers_Driven'] = dataset_merged['Kilometers_Driven'].fillna(median_km)

In [152]:
dataset_merged= dataset_merged[dataset_merged['Kilometers_Driven'] < 300000]
dataset_merged = dataset_merged[dataset_merged['Price'] < 100]
dataset_merged = dataset_merged.drop_duplicates()

In [153]:
# Keep only reasonable mileage records or true new cars
dataset_merged = dataset_merged[~((dataset_merged['Kilometers_Driven'] < 500) & (dataset_merged['Year'] < 2022))]

In [154]:
brand_counts = dataset_merged['Brand'].value_counts()
rare_brands = brand_counts[brand_counts < 10].index
dataset_merged['Brand'] = dataset_merged['Brand'].replace(rare_brands, 'other')

In [155]:
fuel_counts = dataset_merged['Fuel_Type'].value_counts()
rare_fuels = fuel_counts[fuel_counts < 21].index
dataset_merged['Fuel_Type'] = dataset_merged['Fuel_Type'].replace(rare_fuels, 'other')
print(dataset_merged['Owner_Type'].unique())


<StringArray>
['first', 'second', 'fourth & above', 'third']
Length: 4, dtype: str


In [156]:
from sklearn.preprocessing import OrdinalEncoder
o_encoder = OrdinalEncoder(categories=[['fourth & above', 'third', 'second', 'first']])
dataset_merged['Owner_Type'] =o_encoder.fit_transform(dataset_merged[['Owner_Type']])
print(dataset_merged['Owner_Type'].unique())

[3. 2. 0. 1.]


In [157]:
# 1. Step 1: Find the high-frequency models (e.g., appearing 15 or more times)
model_counts = dataset_merged['Model'].value_counts()
top_models = model_counts[model_counts >= 15].index

# 2. Step 2: Keep the top ones, mask everything else as 'Other'
dataset_merged['Model_encoded'] = dataset_merged['Model'].apply(lambda x: x if x in top_models else 'Other')
dataset_merged= dataset_merged.drop(columns=['Model'])


In [158]:
from sklearn.preprocessing import LabelEncoder 
le = LabelEncoder()
dataset_merged['Transmission'] = le.fit_transform(dataset_merged['Transmission']) 


In [159]:
dataset_merged= pd.get_dummies(dataset_merged, columns=['Brand', 'Fuel_Type','Model_encoded'], drop_first=True)
dataset_merged= dataset_merged.astype(int, errors='ignore')

In [160]:
dataset_merged = dataset_merged[dataset_merged['Price'] >= 0.5]
print(dataset_merged.shape)

(14370, 212)


In [161]:
print(dataset_merged.shape)
dataset_merged.drop_duplicates(inplace=True)
print(dataset_merged.shape)

(14370, 212)
(14222, 212)


In [162]:
(dataset_merged ==0).sum()

Year                                     0
Kilometers_Driven                        0
Transmission                          5629
Owner_Type                               9
Price                                    0
                                     ...  
Model_encoded_i20                    14071
Model_encoded_i20 1.2 Magna          14205
Model_encoded_i20 Active             14206
Model_encoded_i20 Sportz 1.2         14207
Model_encoded_maruti-suzuki-dzire    14188
Length: 212, dtype: int64

In [163]:
cols = list(dataset_merged.columns)
cols.remove('Price')
cols.append('Price')
dataset_merged = dataset_merged[cols]

In [164]:
dataset_merged.to_csv('../data/cleaned_data.csv', index=False)